# Compare the confusable pairs

Show random examples of each class pair, grouped by what should separate the pair.

- **container:** the dish it is served in helps identify the menu.
- **food:** the food itself separates the pair, and the container helps little.

Images come from the training data only, chosen by seed rather than by hand.


In [ ]:
VARIANT = "original"  # original, clahe, histeq, gsam, gsam_crop, gsam_light, gsam_light_crop, sam, sam_crop
N_PER_CLASS = 5       # examples per class
SEED = 0              # change to see other random examples
SAVE_FIGURES = True   # writes PNGs to analyst/figures/


In [ ]:
import os
import random
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

# ---- paths (preset) -------------------------------------------------------
# project root = the folder that contains pipeline_preprocess/ (works from any sub-folder)
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline_preprocess").is_dir())
DATA_DIR = Path(os.environ.get("FOOD_DATA_DIR", PROJECT_ROOT / "data"))
FIGURE_DIR = PROJECT_ROOT / "analyst" / "figures"

PAIRS = [("container", "ramen", "pho"),
         ("container", "chocolate_cake", "chocolate_mousse"),
         ("food", "steak", "filet_mignon"),
         ("food", "fried_rice", "pad_thai"),
         ("food", "caesar_salad", "greek_salad")]


In [ ]:
variant_dir = DATA_DIR / "preprocessing_data" / f"{VARIANT}_data"
paths = defaultdict(list)

if variant_dir.is_dir():
    for path in sorted((variant_dir / "train").glob("*/*.jpg")):
        paths[path.parent.name].append(path)
    source = VARIANT
else:
    # generated data not here yet: use the official train list so test images stay unseen
    assert VARIANT == "original", f"{variant_dir} not found. Unzip it from data_zips/ on Drive."
    raw = DATA_DIR / "raw_data"
    for line in open(raw / "meta" / "meta" / "train.txt"):
        cls, img_id = line.strip().split("/")
        paths[cls].append(raw / "images" / cls / f"{img_id}.jpg")
    source = "raw photos, official train list"

print("source:", source)


In [ ]:
rng = random.Random(SEED)

for group, class_a, class_b in PAIRS:
    fig, axes = plt.subplots(2, N_PER_CLASS, figsize=(2.6 * N_PER_CLASS, 5.8))
    for row, cls in zip(axes, (class_a, class_b)):
        for ax, path in zip(row, rng.sample(paths[cls], N_PER_CLASS)):
            ax.imshow(Image.open(path).convert("RGB"))
            ax.set_xticks([]), ax.set_yticks([])
        row[0].set_ylabel(cls, fontsize=12)
    fig.suptitle(f"{group}: {class_a} vs {class_b}  ({source})", fontsize=13)
    fig.tight_layout()

    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIGURE_DIR / f"pair_{group}_{class_a}_vs_{class_b}_{VARIANT}.png", dpi=120)
    plt.show()
